# Week 4 — Agentic RAG & Evaluation

알고리즘 코딩 테스트 코칭 RAG 를 두 방향으로 발전시킨다.

1. **검색·생성 튜닝** — Hybrid(BM25+RRF) · Reranker(cross-encoder) · LlamaParse · grade_docs+rewrite(CRAG 미니) · hallucination grader(Self-RAG 미니)
2. **정량/정성 평가** — retrieval_eval(4지표 × K 스윕) · answer_eval(LLM-as-Judge) · hop별 집계

핵심 가치는 점수를 올리는 것이 아니라 **RAG 성능을 측정 가능하게 만들고 변화를 근거와 함께 설명하는 것**이다.

> 실행 전 준비: `.env` 에 `OPENAI_API_KEY` (임베딩·생성·judge), 선택적으로 `LLAMA_CLOUD_API_KEY`(PDF 파싱), reranker 로컬 모델용 `torch`/`sentence-transformers` 설치. 인덱스가 없으면 `python -m rag.indexing` 으로 빌드한다(week3 인덱스를 `.cache/` 로 복사해 두면 재임베딩 비용 0).

In [ ]:
import os, sys, json
sys.path.insert(0, os.path.abspath("."))
sys.path.insert(0, os.path.abspath("eval"))
import pandas as pd
pd.set_option("display.precision", 3)

## 0. 인덱스 준비

markdown / recursive 전략별 FAISS 인덱스를 빌드한다. 이미 `.cache/` 에 있으면 건너뛴다.

In [ ]:
from rag import INDEX_DIR
if not (INDEX_DIR / "markdown").exists():
    from rag.indexing import build_all
    build_all()
else:
    print("index exists:", [p.name for p in INDEX_DIR.iterdir()])

## 1. 검색 모드 데모 — vector / hybrid / rerank

같은 쿼리에 대해 세 모드가 어떤 문서를 어떤 순서로 가져오는지 비교한다. (`rerank` 는 `torch`/`sentence-transformers` 필요)

In [ ]:
from rag.retriever import retrieve
q = "입국심사 문제가 왜 parametric search 인가"
for mode in ("vector", "hybrid", "rerank"):
    print(f"\n[{mode}]")
    try:
        for d in retrieve(q, mode=mode, k=5):
            print(" -", d.metadata["source"], "|", d.metadata["chunk_id"])
    except Exception as e:
        print("   (skipped:", type(e).__name__, e, ")")

## 2. Harness 1 — retrieval_eval (정량, 결정적)

vector → +hybrid → +rerank 로 검색이 단계별로 얼마나 올라가는지 4지표 × K 스윕으로 분해한다. (LLM 없음)

In [ ]:
from retrieval_eval import run_retrieval_eval, to_dataframe
ret = run_retrieval_eval()
print("=== 전체 (all) ===")
display(to_dataframe(ret, "all"))

In [ ]:
print("=== single-hop ===");  display(to_dataframe(ret, "single"))
print("=== multi-hop ===");   display(to_dataframe(ret, "multi"))

**해석 포인트** — md 문서가 깨끗해 vector-only Recall@K 가 이미 높을 수 있다(천장 효과). 그래서 순위 민감 지표(MRR, Hit@1)와 multi-hop 행에서 hybrid/rerank 의 차이를 본다.

## 3. 그래프 데모 — baseline vs agentic

같은 질문을 두 그래프로 실행해 최종 답변을 비교한다.

In [ ]:
import graph_baseline, graph_agentic
q = "벨만-포드 알고리즘의 의사코드를 알려줘"  # 도메인 밖: '모른다'고 답해야 함
print("[baseline]\n", json.dumps(graph_baseline.ask(q), ensure_ascii=False, indent=2))
print("\n[agentic]\n", json.dumps(graph_agentic.ask(q), ensure_ascii=False, indent=2))

## 4. Harness 2 — answer_eval (정성, LLM-as-Judge)

baseline vs agentic 의 최종 답변을 Correctness·Groundedness 로 채점한다. grade_docs·rewrite·hallucination grader 의 가치가 여기서 드러난다.

> OpenAI API 호출 비용(gpt-4o-mini, 소액)이 발생한다. 처음에는 `limit` 으로 일부만 돌려 비용을 확인한다.

In [ ]:
from answer_eval import run_answer_eval, to_dataframe as ans_df
ans = run_answer_eval()  # 전체 36문항. 비용이 걱정되면 run_answer_eval(limit=6)
print("=== 전체 (all) ===");    display(ans_df(ans["summary"], "all"))
print("=== single-hop ===");    display(ans_df(ans["summary"], "single"))
print("=== multi-hop ===");     display(ans_df(ans["summary"], "multi"))
print("=== out-of-domain ==="); display(ans_df(ans["summary"], "none"))

In [ ]:
# 실패 모드별로 baseline 대비 agentic 이 무엇을 바꿨는지 개별 레코드로 본다.
for r in ans["records"]:
    b, a = r["baseline"], r["agentic"]
    print(f"[{r['id']}|{r['hop']}|{r['failure_mode']}] "
          f"correct {b['correctness']}->{a['correctness']}  "
          f"ground {b['groundedness']}->{a['groundedness']}")

## 5. (선택) LlamaParse 로 PDF 편입

`LLAMA_CLOUD_API_KEY` 가 있으면 PDF 1개를 markdown 으로 파싱해 `data/parsed/` 에 넣고 인덱스를 재빌드한다.

```python
from parsing.parse_pdf import parse_pdf
parse_pdf("<some.pdf>")
from rag.indexing import build_all; build_all()
```